<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/MIXTRAL_GGUF_BENCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF

In [ ]:
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu \
  --no-cache-dir

## CASE1

In [ ]:
import os
import sys
import time
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# 1. Target repository & cached binary
REPO_ID = "frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF"
FILENAME = "mixtral_8x22b_q4_k_m.gguf"
HF_TOKEN = userdata.get("HF_TOKEN")

print(f"[*] Resolving {FILENAME} from {REPO_ID}...")
model_path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    token=HF_TOKEN
)

# 2. Initialize runtime with optimized CPU & memory settings
print("[*] Loading Mixtral-8x22B MoE into memory...")
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=8,              # Locked to physical execution cores
    n_threads_batch=8,
    n_gpu_layers=0,           # Pure CPU execution
    use_mlock=True,           # Pins MoE expert weights in RAM
    use_mmap=True,
    flash_attn=True,          # Reduces CPU memory operations
    type_k=8,                 # Quantized KV cache (Q8_0)
    type_v=8,
    last_n_tokens_size=128,
    logits_all=False,
    seed=123,                 # Deterministic seed locking
    verbose=False
)

# 3. Direct prompt
prompt = "[INST] Define the LEFM operator under AST. [/INST]"

# 4. Streaming execution with timing diagnostics
print("\n--- Model Output (Streaming) ---")
sys.stdout.flush()

stream = llm(
    prompt=prompt,
    max_tokens=256,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.15,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123,
    stream=True               # Enables token-by-token generator
)

token_count = 0
collected_tokens = []
start_time = None
finish_reason = None

for chunk in stream:
    if start_time is None:
        start_time = time.perf_counter()  # Excludes prefill / prompt ingestion time

    choice = chunk["choices"][0]
    token_text = choice.get("text", "")
    finish_reason = choice.get("finish_reason")

    if token_text:
        token_count += 1
        collected_tokens.append(token_text)
        sys.stdout.write(token_text)
        sys.stdout.flush()

end_time = time.perf_counter()
total_gen_time = end_time - start_time if start_time else 0.0
throughput = (token_count / total_gen_time) if total_gen_time > 0 else 0.0

# 5. Clean, enforce formatting, and report throughput
raw_text = "".join(collected_tokens).strip()
if raw_text.startswith("Assistant:"):
    raw_text = raw_text[len("Assistant:"):].strip()
single_response = raw_text.split("Assistant:")[0].strip()

print("\n\n--------------------")
print(f"Finish Reason:       {finish_reason}")
print(f"Generated Tokens:    {token_count}")
print(f"Generation Time:     {total_gen_time:.2f} s")
print(f"Generation Speed:    {throughput:.2f} tokens/sec")
print("--------------------")

[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...


mixtral_8x22b_q4_k_m.gguf: reconstructing file:   0%|          |  0.00B / 85.6GB            

mixtral_8x22b_q4_k_m.gguf: downloading bytes:           |  0.00B            

[*] Loading Mixtral-8x22B MoE into memory...

--- Model Output (Streaming) ---

Assistant: The operator is tied strictly to the AST, and what it does. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations in complex analysis. Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, because R_KERNEL = [2, 3, 5, 7, 11, 13]. By anchoring the network state space via Topo-CBP to prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13] the LEFM operator bypasses this limitation, embedding the network state space into the discrete prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13], avoiding the trap of continuous integration. This permanently precludes any asymptotic tail approximations over infinite domains.
Assistant: Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, because R_KERNEL = [2, 3, 5, 7, 11, 13]. By anchoring the network state 

## CASE2

In [ ]:
import sys
import time

# 1. Do NOT put "Assistant:" in stop—it kills generation on token 1
stream = llm(
    prompt=prompt,
    max_tokens=256,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.18,
    stop=["</s>", "[INST]", "\n[INST]"],  # Clean stops only
    seed=123,
    stream=True
)

token_count = 0
collected_tokens = []
start_time = None
finish_reason = None
buffer = ""
assistant_prefix_stripped = False

for chunk in stream:
    choice = chunk["choices"][0]
    token_text = choice.get("text", "")
    finish_reason = choice.get("finish_reason")

    if not token_text:
        continue

    # Start timing on the first valid token received
    if start_time is None:
        start_time = time.perf_counter()

    # Check for runaway repeating loops later in generation
    if token_count > 15 and ("Assistant:" in token_text or "\nAssistant:" in token_text):
        finish_reason = "stop"
        break

    # Strip the leading "Assistant:" prompt tag if the model emits it at the start
    if not assistant_prefix_stripped:
        buffer += token_text
        if len(buffer) < len("Assistant: "):
            # Wait until we have enough characters to see if it's the prefix
            continue
        else:
            if buffer.strip().startswith("Assistant:"):
                # Remove the leading prefix
                clean_initial = buffer.strip()[len("Assistant:"):].lstrip()
                buffer = clean_initial
            assistant_prefix_stripped = True
            if buffer:
                token_count += 1
                collected_tokens.append(buffer)
                sys.stdout.write(buffer)
                sys.stdout.flush()
            continue

    token_count += 1
    collected_tokens.append(token_text)
    sys.stdout.write(token_text)
    sys.stdout.flush()

end_time = time.perf_counter()
total_gen_time = end_time - start_time if start_time else 0.0
throughput = (token_count / total_gen_time) if total_gen_time > 0 else 0.0

print("\n\n--------------------")
print(f"Finish Reason:       {finish_reason}")
print(f"Generated Tokens:    {token_count}")
print(f"Generation Time:     {total_gen_time:.2f} s")
print(f"Generation Speed:    {throughput:.2f} tokens/sec")
print("--------------------")

 The operator is tied strictly to the AST, and what it does in detail. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations with asymptotic bounds when addressing the Riemann Hypothesis without stochastic guessing. Classical Hadamard product expansions rely on infinite sets, continuous integration approximations over infinite domains, and deterministically exacted prime-indexed zero drift. By anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13]. The LEFM operator bypasses this limitation by replacing continuous integration approximation with structural determinism.
Assistant: Classical Hadamard product expansions in complex analysis rely on asymptotic tail bounds when addressing the Riemann Hypothesis without stochastic guessing. By anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13], the Laplace-Euler-Fourier-Mellin (

## CASE3

In [ ]:
import sys
import time

# 1. Native stop tokens only.
# Do not place conversational role headers in 'stop'—handle them in the stream loop.
stream = llm(
    prompt=prompt,
    max_tokens=512,
    temperature=0.0,
    top_k=1,
    top_p=1.0,
    repeat_penalty=1.18,
    stop=["</s>", "[INST]", "\n[INST]"],
    seed=123,
    stream=True
)

token_count = 0
collected_tokens = []
start_time = None
finish_reason = None

# Prefix stripping state
prefix_target = "Assistant:"
prefix_buffer = ""
prefix_stripped = False

print("\n--- Model Output (Streaming) ---")
sys.stdout.flush()

for chunk in stream:
    choice = chunk["choices"][0]
    token_text = choice.get("text", "")
    finish_reason = choice.get("finish_reason")

    if not token_text:
        continue

    # Start timing on the exact arrival of the first token
    if start_time is None:
        start_time = time.perf_counter()

    token_count += 1

    # Stage A: Strip leading "Assistant:" safely across variable token boundaries
    if not prefix_stripped:
        prefix_buffer += token_text

        # If buffer starts with target, wait until we have the full header + space
        if prefix_buffer.lstrip().startswith(prefix_target):
            # Check if we passed the full prefix
            idx = prefix_buffer.find(prefix_target)
            after_prefix = prefix_buffer[idx + len(prefix_target):]
            if after_prefix.startswith(" ") or len(after_prefix) > 0:
                clean_text = after_prefix.lstrip()
                prefix_stripped = True
                if clean_text:
                    collected_tokens.append(clean_text)
                    sys.stdout.write(clean_text)
                    sys.stdout.flush()
            continue
        elif len(prefix_buffer.strip()) >= len(prefix_target):
            # The model didn't start with "Assistant:", flush everything buffered
            prefix_stripped = True
            collected_tokens.append(prefix_buffer)
            sys.stdout.write(prefix_buffer)
            sys.stdout.flush()
            continue
        else:
            # Accumulating first few characters to check prefix
            continue

    # Stage B: Catch runaway loops (e.g., subsequent "Assistant:" headers)
    # Check the recent trailing window to handle split-chunk arrivals
    collected_tokens.append(token_text)
    tail_text = "".join(collected_tokens[-6:])
    if token_count > 20 and ("Assistant:" in tail_text or "\nAssistant:" in tail_text):
        finish_reason = "stop"
        # Erase the echoed header from console if partially written
        break

    sys.stdout.write(token_text)
    sys.stdout.flush()

end_time = time.perf_counter()
total_gen_time = end_time - start_time if start_time else 0.0
throughput = (token_count / total_gen_time) if total_gen_time > 0 else 0.0

print("\n\n--------------------")
print(f"Finish Reason:       {finish_reason}")
print(f"Generated Tokens:    {token_count}")
print(f"Generation Time:     {total_gen_time:.2f} s")
print(f"Generation Speed:    {throughput:.2f} tokens/sec")
print("--------------------")


--- Model Output (Streaming) ---
The operator is tied strictly to the AST, and what it does in detail. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations with asymptotic bounds when addressing the Riemann Hypothesis without stochastic guessing. Classical Hadamard product expansions rely on infinite sets, continuous integration approximations over infinite domains, and deterministically exacted prime-indexed zero drift. By anchoring the network state space via Topo-CBP to the discrete prime kernel R_KERNEL = [2, 3, 5, 7, 11, 13]. The LEFM operator bypasses this limitation by replacing continuous integration approximation with structural determinism.
Assistant

--------------------
Finish Reason:       stop
Generated Tokens:    160
Generation Time:     31.41 s
Generation Speed:    5.09 tokens/sec
--------------------


## CASE4

In [ ]:
import os
import sys
import time
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama


class GGUFEngine:
    """Standardized bare-metal CPU streaming runtime and benchmark harness for quantized GGUF architectures."""
    def __init__(
        self,
        model_path: str,
        n_ctx: int = 4096,
        n_threads: int = 8,
        use_mlock: bool = True,
        use_mmap: bool = True,
        flash_attn: bool = True,
        cache_type_k: int = 8,
        cache_type_v: int = 8,
        seed: int = 123,
        verbose: bool = False,
    ):
        print(f"[*] Initializing GGUF Engine: {os.path.basename(model_path)}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=n_ctx,
            n_threads=n_threads,
            n_threads_batch=n_threads,
            n_gpu_layers=0,
            use_mlock=use_mlock,
            use_mmap=use_mmap,
            flash_attn=flash_attn,
            type_k=cache_type_k,
            type_v=cache_type_v,
            last_n_tokens_size=128,
            logits_all=False,
            seed=seed,
            verbose=verbose,
        )
        self.seed = seed

    @classmethod
    def from_hub(
        cls,
        repo_id: str,
        filename: str,
        token: str = None,
        **kwargs,
    ) -> "GGUFEngine":
        print(f"[*] Resolving {filename} from {repo_id}...")
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            token=token or os.environ.get("HF_TOKEN"),
        )
        return cls(model_path=path, **kwargs)

    def stream_generate(
        self,
        prompt: str,
        max_tokens: int = 512,
        temperature: float = 0.0,
        top_k: int = 1,
        top_p: float = 1.0,
        repeat_penalty: float = 1.18,
        prefix_target: str = "Assistant:",
    ) -> dict:
        stream = self.llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repeat_penalty=repeat_penalty,
            stop=["</s>", "[INST]", "\n[INST]"],
            seed=self.seed,
            stream=True,
        )

        token_count = 0
        collected_tokens = []
        start_time = None
        finish_reason = None
        prefix_buffer = ""
        prefix_stripped = False

        print("\n--- Model Output (Streaming) ---")
        sys.stdout.flush()

        for chunk in stream:
            choice = chunk["choices"][0]
            token_text = choice.get("text", "")
            finish_reason = choice.get("finish_reason")

            if not token_text:
                continue

            if start_time is None:
                start_time = time.perf_counter()

            token_count += 1

            if not prefix_stripped:
                prefix_buffer += token_text
                if prefix_buffer.lstrip().startswith(prefix_target):
                    idx = prefix_buffer.find(prefix_target)
                    after_prefix = prefix_buffer[idx + len(prefix_target):]
                    if after_prefix.startswith(" ") or len(after_prefix) > 0:
                        clean_text = after_prefix.lstrip()
                        prefix_stripped = True
                        if clean_text:
                            collected_tokens.append(clean_text)
                            sys.stdout.write(clean_text)
                            sys.stdout.flush()
                    continue
                elif len(prefix_buffer.strip()) >= len(prefix_target):
                    prefix_stripped = True
                    collected_tokens.append(prefix_buffer)
                    sys.stdout.write(prefix_buffer)
                    sys.stdout.flush()
                    continue
                else:
                    continue

            collected_tokens.append(token_text)
            tail_text = "".join(collected_tokens[-6:])
            if token_count > 20 and (prefix_target in tail_text or f"\n{prefix_target}" in tail_text):
                finish_reason = "stop"
                sys.stdout.write("\b" * len(token_text))
                sys.stdout.write(" " * len(token_text))
                sys.stdout.write("\b" * len(token_text))
                sys.stdout.flush()
                break

            sys.stdout.write(token_text)
            sys.stdout.flush()

        end_time = time.perf_counter()
        total_time = end_time - start_time if start_time else 0.0
        tps = (token_count / total_time) if total_time > 0 else 0.0

        metrics = {
            "text": "".join(collected_tokens).strip(),
            "finish_reason": finish_reason,
            "tokens_generated": token_count,
            "generation_time_sec": total_time,
            "tokens_per_sec": tps,
        }

        print("\n\n--------------------")
        print(f"Finish Reason:       {metrics['finish_reason']}")
        print(f"Generated Tokens:    {metrics['tokens_generated']}")
        print(f"Generation Time:     {metrics['generation_time_sec']:.2f} s")
        print(f"Generation Speed:    {metrics['tokens_per_sec']:.2f} tokens/sec")
        print("--------------------")

        return metrics


# Execution block
HF_TOKEN = userdata.get("HF_TOKEN")

engine = GGUFEngine.from_hub(
    repo_id="frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF",
    filename="mixtral_8x22b_q4_k_m.gguf",
    token=HF_TOKEN,
    n_ctx=4096,
    n_threads=8,
    seed=123,
)

metrics = engine.stream_generate(
    prompt="[INST] Define the LEFM operator under AST. [/INST]",
    max_tokens=512,
)

[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Initializing GGUF Engine: mixtral_8x22b_q4_k_m.gguf

--- Model Output (Streaming) ---
The operator is tied strictly to the AST, and what it does. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations in complex analysis. Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, because R_KERNEL = [2, 3, 5, 7, 11, 13]. By anchoring the network state space via Topo-CBP to prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13] the LEFM operator bypasses this limitation, embedding the exact same mathematical structure required to stabilize the LEFM operator. This permanently precludes any rogue zero drift seen in 20th C complex analysis methods.
Assistant 

--------------------
Finish Reason:       stop
Generated Tokens:    179
Generation Time:     34.84 s
Generation Speed:    5.

## CASE5

In [ ]:
import os
import sys
import time
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

from google.colab import userdata
from huggingface_hub import hf_hub_download
from llama_cpp import Llama


class GGUFEngine:
    """Standardized bare-metal CPU streaming runtime and benchmark harness for quantized GGUF architectures."""
    def __init__(
        self,
        model_path: str,
        n_ctx: int = 4096,
        n_threads: int = 8,
        use_mlock: bool = True,
        use_mmap: bool = True,
        flash_attn: bool = True,
        cache_type_k: int = 8,
        cache_type_v: int = 8,
        seed: int = 123,
        verbose: bool = False,
    ):
        print(f"[*] Initializing GGUF Engine: {os.path.basename(model_path)}")
        self.llm = Llama(
            model_path=model_path,
            n_ctx=n_ctx,
            n_threads=n_threads,
            n_threads_batch=n_threads,
            n_gpu_layers=0,
            use_mlock=use_mlock,
            use_mmap=use_mmap,
            flash_attn=flash_attn,
            type_k=cache_type_k,
            type_v=cache_type_v,
            last_n_tokens_size=128,
            logits_all=False,
            seed=seed,
            verbose=verbose,
        )
        self.seed = seed

    @classmethod
    def from_hub(
        cls,
        repo_id: str,
        filename: str,
        token: str = None,
        **kwargs,
    ) -> "GGUFEngine":
        print(f"[*] Resolving {filename} from {repo_id}...")
        path = hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            token=token or os.environ.get("HF_TOKEN"),
        )
        return cls(model_path=path, **kwargs)

    def stream_generate(
        self,
        prompt: str,
        max_tokens: int = 512,
        temperature: float = 0.0,
        top_k: int = 1,
        top_p: float = 1.0,
        repeat_penalty: float = 1.18,
        prefix_target: str = "Assistant:",
    ) -> dict:
        stream = self.llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repeat_penalty=repeat_penalty,
            stop=["</s>", "[INST]", "\n[INST]"],
            seed=self.seed,
            stream=True,
        )

        token_count = 0
        collected_tokens = []
        start_time = None
        finish_reason = None
        prefix_buffer = ""
        prefix_stripped = False

        print("\n--- Model Output (Streaming) ---")
        sys.stdout.flush()

        for chunk in stream:
            choice = chunk["choices"][0]
            token_text = choice.get("text", "")
            finish_reason = choice.get("finish_reason")

            if not token_text:
                continue

            if start_time is None:
                start_time = time.perf_counter()

            token_count += 1

            # Stage A: Strip leading prompt tag without printing it
            if not prefix_stripped:
                prefix_buffer += token_text
                if prefix_buffer.lstrip().startswith(prefix_target):
                    idx = prefix_buffer.find(prefix_target)
                    after_prefix = prefix_buffer[idx + len(prefix_target):]
                    if after_prefix.startswith(" ") or len(after_prefix) > 0:
                        clean_text = after_prefix.lstrip()
                        prefix_stripped = True
                        if clean_text:
                            collected_tokens.append(clean_text)
                            sys.stdout.write(clean_text)
                            sys.stdout.flush()
                    continue
                elif len(prefix_buffer.strip()) >= len(prefix_target):
                    prefix_stripped = True
                    collected_tokens.append(prefix_buffer)
                    sys.stdout.write(prefix_buffer)
                    sys.stdout.flush()
                    continue
                else:
                    continue

            # Stage B: Intercept conversational loop echo and purge partial token artifacts
            collected_tokens.append(token_text)
            tail_text = "".join(collected_tokens[-6:])

            # Check for full target or any trailing partial prefix (e.g., 'Assistan', 'Assist')
            found_echo = False
            for length in range(len(prefix_target), 3, -1):
                sub = prefix_target[:length]
                if token_count > 20 and (sub in tail_text or f"\n{sub}" in tail_text):
                    found_echo = True
                    break

            if found_echo:
                finish_reason = "stop"
                # Strip the partial echo string from collected text
                collected_tokens.pop()
                # Erase the echoed characters completely from terminal output
                sys.stdout.write("\b" * len(token_text))
                sys.stdout.write(" " * len(token_text))
                sys.stdout.write("\b" * len(token_text))
                sys.stdout.flush()
                break

            sys.stdout.write(token_text)
            sys.stdout.flush()

        end_time = time.perf_counter()
        total_time = end_time - start_time if start_time else 0.0
        tps = (token_count / total_time) if total_time > 0 else 0.0

        metrics = {
            "text": "".join(collected_tokens).strip(),
            "finish_reason": finish_reason,
            "tokens_generated": token_count,
            "generation_time_sec": total_time,
            "tokens_per_sec": tps,
        }

        print("\n\n--------------------")
        print(f"Finish Reason:       {metrics['finish_reason']}")
        print(f"Generated Tokens:    {metrics['tokens_generated']}")
        print(f"Generation Time:     {metrics['generation_time_sec']:.2f} s")
        print(f"Generation Speed:    {metrics['tokens_per_sec']:.2f} tokens/sec")
        print("--------------------")

        return metrics


# Execution block
HF_TOKEN = userdata.get("HF_TOKEN")

engine = GGUFEngine.from_hub(
    repo_id="frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF",
    filename="mixtral_8x22b_q4_k_m.gguf",
    token=HF_TOKEN,
    n_ctx=4096,
    n_threads=8,
    seed=123,
)

metrics = engine.stream_generate(
    prompt="[INST] Define the LEFM operator under AST. [/INST]",
    max_tokens=512,
)

[*] Resolving mixtral_8x22b_q4_k_m.gguf from frankmorales2020/mixtral-8x22b-instruct-v0.1-topo-cbp-GGUF...
[*] Initializing GGUF Engine: mixtral_8x22b_q4_k_m.gguf

--- Model Output (Streaming) ---
The operator is tied strictly to the AST, and what it does. This provides a mathematical breakdown of how the LEFM operator bypasses classical Hadamard product limitations in complex analysis. Classical Hadamard product expansions rely on infinite asymptotic tail approximations over infinite domains, because R_KERNEL = [2, 3, 5, 7, 11, 13]. By anchoring the network state space via Topo-CBP to prime kerneled R_KERNEL = [2, 3, 5, 7, 11, 13] the LEFM operator bypasses this limitation, embedding the exact same mathematical structure required to stabilize the LEFM operator. This permanently precludes any rogue zero drift seen in 20th C complex analysis methods.
Ass      

--------------------
Finish Reason:       stop
Generated Tokens:    178
Generation Time:     35.12 s
Generation Spe